In [5]:
from sklearn.ensemble import VotingClassifier, StackingClassifier, AdaBoostClassifier
import ast

import colorcet as cc
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import ListedColormap
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.calibration import CalibratedClassifierCV
np.random.seed(42)



In [6]:
# Reload the preprocessed data
df_train = pd.read_pickle("../../data/processed/df_train_preprocessed.pkl")
df_val = pd.read_pickle("../../data/processed/df_val_preprocessed.pkl")

label_encoder = LabelEncoder().fit(df_train.industry)

X_train = np.array(df_train["business_description_embedding"].tolist())
y_train = label_encoder.transform(df_train.industry)

X_val = np.array(df_val["business_description_embedding"].tolist())
y_val = label_encoder.transform(df_val.industry)


In [7]:
# Make a soft voting ensemble classifier of calibrated classifiers using different models.

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

models = [
    (
        "logistic",
        make_pipeline(
            StandardScaler(),
            LogisticRegression(C=1.0, max_iter=1000, random_state=42),
        ),
    ),
    (
        "mlp",
        make_pipeline(
            StandardScaler(),
            MLPClassifier(
                hidden_layer_sizes=(128,),
                alpha=1e-3,
                batch_size=512,
                early_stopping=True,
                max_iter=100,
                random_state=42,
            ),
        ),
    ),
    (
        "hist_gradient_boosting",
        HistGradientBoostingClassifier(
            max_iter=150,
            max_leaf_nodes=15,
            min_samples_leaf=50,
            l2_regularization=1.0,
            early_stopping=True,
            random_state=42,
        ),
    ),
]

calibrated_models = [
    (
        name,
        CalibratedClassifierCV(
            estimator=model,
            method="sigmoid",
            cv=3,
            ensemble=False,
        ),
    )
    for name, model in models
]

soft_voting = VotingClassifier(
    estimators=calibrated_models,
    voting="soft",
    n_jobs=1,
)

soft_voting.fit(X_train, y_train)

,"estimators estimators: list of (str, estimator) tuplesInvoking the ``fit`` method on the ``VotingClassifier`` will fit clonesof those original estimators that will be stored in the class attribute``self.estimators_``. An estimator can be set to ``'drop'`` using:meth:`set_params`... versionchanged:: 0.21 ``'drop'`` is accepted. Using None was deprecated in 0.22 and support was removed in 0.24.","[('logistic', ...), ('mlp', ...), ...]"
,"voting voting: {'hard', 'soft'}, default='hard'If 'hard', uses predicted class labels for majority rule voting.Else if 'soft', predicts the class label based on the argmax ofthe sums of the predicted probabilities, which is recommended foran ensemble of well-calibrated classifiers.",'soft'
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for ``fit``.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionadded:: 0.18",1
,"weights weights: array-like of shape (n_classifiers,), default=NoneSequence of weights (`float` or `int`) to weight the occurrences ofpredicted class labels (`hard` voting) or class probabilitiesbefore averaging (`soft` voting). Uses uniform weights if `None`.",None
,"flatten_transform flatten_transform: bool, default=TrueAffects shape of transform output only when voting='soft'If voting='soft' and flatten_transform=True, transform method returnsmatrix with shape (n_samples, n_classifiers * n_classes). Ifflatten_transform=False, it returns(n_classifiers, n_samples, n_classes).",True
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting will be printed as itis completed... versionadded:: 0.23",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels.","ndarray[int64](25,)","[ 0, 1, 2,...,22,23,24]"
"estimators_ estimators_: list of classifiersThe collection of fitted sub-estimators as defined in ``estimators``that are not 'drop'. Note that sub-estimators are always fitted oninteger-encoded labels (see ``le_`` attribute). When ``y`` containsnon-integer class labels (e.g. strings), use ``le_.inverse_transform``to map predictions back to the original label space.",list,"[CalibratedCla..._state=42))])), CalibratedCla..._state=42))])), CalibratedCla...dom_state=42))]"
le_ le_: :class:`~sklearn.preprocessing.LabelEncoder`Transformer used to encode the labels during fit and decode duringprediction. Sub-estimators in ``estimators_`` are fitted on theinteger-encoded labels produced by this encoder.,LabelEncoder,LabelEncoder()
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying classifier exposes such an attribute when fit... versionadded:: 0.24,int,768


In [8]:
def confidence_score(y_true, y_pred, confidence):
    """
    Score for the confidence weighted prediction task.

    Parameters:
    -----------
    y_true : array-like
        True class labels
    y_pred : array-like
        Predicted class labels
    confidence : array-like
        Confidence for each prediction, in [0, 1]
        (= estimated probability that the prediction is correct)

    Returns:
    --------
    float
        weighted F1 * (1 - Brier score), between 0 and 1 (higher is better)
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    confidence = np.asarray(confidence, dtype=float)

    assert len(y_true) == len(y_pred) == len(confidence), "Inputs must have the same length"
    assert np.all((confidence >= 0) & (confidence <= 1)), "Confidence must be between 0 and 1"

    f1 = f1_score(y_true, y_pred, average="weighted")
    correct = (y_true == y_pred).astype(float)
    brier = np.mean((confidence - correct) ** 2)

    return f1 * (1 - brier)

In [ ]:
def evaluate_model(model, X_val, y_val):
        
    proba = model.predict_proba(X_val)
    y_pred = model.classes_[proba.argmax(axis=1)]
    confidence = proba.max(axis=1)

    correct = (y_pred == y_val).astype(float)
    f1 = f1_score(y_val, y_pred, average="weighted")
    brier = np.mean((confidence - correct) ** 2)

    print("Weighted F1:", f1)
    print("Brier:", brier)
    print("Combined score:", f1 * (1 - brier))
    print("Mean confidence:", confidence.mean())
    print("Accuracy:", correct.mean())




    confidence = proba.max(axis=1)
    correct = y_pred == y_val

    print("\nMean confidence:", confidence.mean())
    print("Accuracy:", correct.mean())

    # Compare confidence with observed accuracy within each bin.
    calibration = pd.DataFrame({
        "confidence": confidence,
        "correct": correct,
    })

    calibration["bin"] = pd.cut(
        calibration["confidence"],
        bins=np.linspace(0, 1, 11),
        include_lowest=True,
    )

    print(
        calibration.groupby("bin", observed=True).agg(
            count=("correct", "size"),
            mean_confidence=("confidence", "mean"),
            accuracy=("correct", "mean"),
        )
    )

In [19]:
evaluate_model(soft_voting, X_val, y_val)

Weighted F1: 0.7469780168544808
Brier: 0.18017498784631467
Combined score: 0.6123912617462605
Mean confidence: 0.586173588497615
Accuracy: 0.7500921489126429

Mean confidence: 0.586173588497615
Accuracy: 0.7500921489126429
            count  mean_confidence  accuracy
bin                                         
(0.1, 0.2]     28         0.170321  0.214286
(0.2, 0.3]    121         0.258522  0.363636
(0.3, 0.4]    263         0.355081  0.456274
(0.4, 0.5]    363         0.449233  0.592287
(0.5, 0.6]    469         0.549680  0.716418
(0.6, 0.7]    701         0.650733  0.853067
(0.7, 0.8]    620         0.746457  0.927419
(0.8, 0.9]    148         0.817655  0.952703


Model is underconfident. Need to calibrate final classifier

In [16]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator  # scikit-learn >= 1.6
from sklearn.model_selection import train_test_split

# Preserve class proportions in both halves.
X_cal, X_eval, y_cal, y_eval = train_test_split(
    X_val,
    y_val,
    test_size=0.5,
    stratify=y_val,
    random_state=42,
)

# adjust probabilities.
calibrated_voting = CalibratedClassifierCV(
    estimator=FrozenEstimator(soft_voting),
    method="sigmoid",
)

calibrated_voting.fit(X_cal, y_cal)

,"estimator estimator: estimator instance, default=NoneThe classifier whose output need to be calibrated to provide moreaccurate `predict_proba` outputs. The default classifier isa :class:`~sklearn.svm.LinearSVC`... versionadded:: 1.2",FrozenEstimat...oting='soft'))
,"method method: {'sigmoid', 'isotonic', 'temperature'}, default='sigmoid'The method to use for calibration. Can be:- 'sigmoid', which corresponds to Platt's method (i.e. a binary logistic regression model).- 'isotonic', which is a non-parametric approach.- 'temperature', temperature scaling.Sigmoid and isotonic calibration methods natively support only binaryclassifiers and extend to multi-class classification using a One-vs-Rest (OvR)strategy with post-hoc renormalization, i.e., adjusting the probabilities aftercalibration to ensure they sum up to 1.In contrast, temperature scaling naturally supports multi-class calibration byapplying `softmax(classifier_logits/T)` with a value of `T` (temperature)that optimizes the log loss.For very uncalibrated classifiers on very imbalanced datasets, sigmoidcalibration might be preferred because it fits an additional interceptparameter. This helps shift decision boundaries appropriately when theclassifier being calibrated is biased towards the majority class.Isotonic calibration is not recommended when the number of calibration samplesis too low ``(≪1000)`` since it then tends to overfit... versionchanged:: 1.8 Added option 'temperature'.",'sigmoid'
,"cv cv: int, cross-validation generator, or iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross-validation,- integer, to specify the number of folds,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if ``y`` is binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used. If ``y`` isneither binary nor multiclass, :class:`~sklearn.model_selection.KFold`is used.Refer to the :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors.Base estimator clones are fitted in parallel across cross-validationiterations.See :term:`Glossary <n_jobs>` for more details... versionadded:: 0.24",None
,"ensemble ensemble: bool, or ""auto"", default=""auto""Determines how the calibrator is fitted.""auto"" will use `False` if the `estimator` is a:class:`~sklearn.frozen.FrozenEstimator`, and `True` otherwise.If `True`, the `estimator` is fitted using training data, andcalibrated using testing data, for each `cv` fold. The final estimatoris an ensemble of `n_cv` fitted classifier and calibrator pairs, where`n_cv` is the number of cross-validation folds. The output is theaverage predicted probabilities of all pairs.If `False`, `cv` is used to compute unbiased predictions, via:func:`~sklearn.model_selection.cross_val_predict`, which are thenused for calibration. At prediction time, the classifier used is the`estimator` trained on all the data.Note that this method is also internally implemented in:mod:`sklearn.svm` estimators with the `probabilities=True` parameter... versionadded:: 0.24.. versionchanged:: 1.6 `""auto""` option is added and is the default.",'auto'
Name,Type,Value
"calibrated_classifiers_ calibrated_classifiers_: list (len() equal to cv or 1 if `ensemble=False`)The list of classifier and calibrator pairs.- When `ensemble=True`, `n_cv` fitted `estimator` and calibrator pairs. `n_cv` is the number of cross-validation folds.- When `ensemble=False`, the `estimator`, fitted on all the data, and fitted calibrator... versionchanged:: 0.24 Single calibrated classifier case when `ensemble=False`.",list,[<sklearn.cali...x778a3c81ce50>]
"

In [20]:
evaluate_model(calibrated_voting, X_eval, y_eval)

Weighted F1: 0.7531698888011745
Brier: 0.1608267161880741
Combined score: 0.6320400488535447
Mean confidence: 0.7089282065932904
Accuracy: 0.7546057479734709

Mean confidence: 0.7089282065932904
Accuracy: 0.7546057479734709
            count  mean_confidence  accuracy
bin                                         
(0.1, 0.2]     10         0.144836  0.200000
(0.2, 0.3]     38         0.256087  0.289474
(0.3, 0.4]     52         0.352080  0.480769
(0.4, 0.5]     86         0.455428  0.546512
(0.5, 0.6]     83         0.549706  0.626506
(0.6, 0.7]    127         0.650868  0.535433
(0.7, 0.8]    396         0.764946  0.755051
(0.8, 0.9]    565         0.817976  0.920354


In [21]:
def evaluate_confidence(model, X, y):
    probabilities = model.predict_proba(X)
    predictions = model.classes_[probabilities.argmax(axis=1)]
    confidence = probabilities.max(axis=1)

    correct = (predictions == np.asarray(y)).astype(float)
    f1 = f1_score(y, predictions, average="weighted")
    brier = np.mean((confidence - correct) ** 2)

    return {
        "weighted_f1": f1,
        "accuracy": correct.mean(),
        "mean_confidence": confidence.mean(),
        "brier": brier,
        "combined_score": f1 * (1 - brier),
    }


comparison = pd.DataFrame({
    "before": evaluate_confidence(soft_voting, X_eval, y_eval),
    "after": evaluate_confidence(calibrated_voting, X_eval, y_eval),
}).T

print(comparison.round(4))

        weighted_f1  accuracy  mean_confidence   brier  combined_score
before       0.7502    0.7524           0.5849  0.1807          0.6146
after        0.7532    0.7546           0.7089  0.1608          0.6320


In [27]:
lr =LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
evaluate_confidence(lr, X_eval, y_eval)



{'weighted_f1': 0.7415471999874749,
 'accuracy': np.float64(0.7450257921886514),
 'mean_confidence': np.float64(0.6823285277057626),
 'brier': np.float64(0.16039217312319345),
 'combined_score': np.float64(0.6226088331080645)}

In [28]:

temperature_lr = CalibratedClassifierCV(
    estimator=FrozenEstimator(lr),
    method="temperature",
)
temperature_lr.fit(X_train, y_train)
evaluate_confidence(temperature_lr, X_eval, y_eval)



{'weighted_f1': 0.7415471999874749,
 'accuracy': np.float64(0.7450257921886514),
 'mean_confidence': np.float64(0.7544943134046471),
 'brier': np.float64(0.15573041199116433),
 'combined_score': np.float64(0.6260657490225311)}

try different calibration method. we still underestimate a bit. 

In [26]:
temperature_voting = CalibratedClassifierCV(
    estimator=FrozenEstimator(soft_voting),
    method="temperature",
)

temperature_voting.fit(X_cal, y_cal)

comparison = pd.DataFrame({
    "original": evaluate_confidence(soft_voting, X_eval, y_eval),
    "sigmoid": evaluate_confidence(calibrated_voting, X_eval, y_eval),
    "temperature": evaluate_confidence(temperature_voting, X_eval, y_eval),
}).T

print(comparison.round(4))

             weighted_f1  accuracy  mean_confidence   brier  combined_score
original          0.7502    0.7524           0.5849  0.1807          0.6146
sigmoid           0.7532    0.7546           0.7089  0.1608          0.6320
temperature       0.7502    0.7524           0.7474  0.1530          0.6354
